In [ ]:
# @title Setup — Run this first
import os

if not os.path.exists('/content/MS_spectra_encoding'):
    !git clone https://github.com/Alexander-Sol/MS_spectra_encoding.git /content/MS_spectra_encoding

%cd /content/MS_spectra_encoding

# Install packages not pre-installed on Colab
!pip install -q pyteomics==4.6.1 spectrum-utils==0.4.2 rapidhash==0.1.0 mmh3

!gdown 1uBQSqXOQduhW9Yy94VRWuV9k3CUR6QbG -O Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML


# 3. Method 2: De Novo Sequencing with Transformers (Casanovo)

## Goals:

In this notebook, we'll be exploring the following three core concepts:

1. [Why de novo sequencing is needed, and what happens when there's no database to search against.](#31-what-if-theres-no-database)

2. [How positional encoding translates raw m/z values into rich, multi-scale vector representations.](#34-positional-encoding)

3. [How Casanovo's Transformer architecture (encoder, decoder, and beam search) predicts full peptide sequences directly from spectra.](#311-casanovos-transformer-architecture)

In [ ]:
# @title Run this cell to import all necessary packages

import numpy as np
import matplotlib.pyplot as plt
from util import *
import math

## 3.1 What if there's no database?

In the previous notebook, we used spectrum binning + feature hashing to cluster spectra and match them against a database. But what happens when:
- You're studying a new organism with no reference proteome?
- You want to discover novel peptides or variants not in any database?
- You need to identify antibodies or other highly variable sequences?

How do you identify peptides then?

Remember that when we use databases, someone has already done the hard work of sequencing the proteins and translating them into peptide sequences. We just match spectra to these known sequences.

De novo sequencing aims to solve all of these problems by predicting the peptide sequence directly from the spectrum, no database required.

## 3.2 Translation Analogy

I'll connect it back to Mass Spectrometry in just a second, but consider this problem. The challenge of making machine learning models for the purpose of translation is quite an old one (in fact Google Translate is an example).

Suppose we are trying to:

1. Take in a paragraph of English text
2. Spit out a corresponding paragraph of French text

That's it! Recent advancements in the field of mass spectrometry have found a way to implement this analogy between mass spectra and their corresponding peptide sequences too. The translation here would be:

1. Take in a spectrum
2. Spit out the corresponding peptides for this spectrum

This is a form of *De novo sequencing*, since we don't have a database to compare anything to. We're simply "predicting" identifications without any outside knowledge.

I will mention though, that in the last notebook we represented *entire spectra* as single fixed-size vectors. Here, the translation model reads a spectrum *peak by peak*. Each individual peak gets its own numerical representation, where we can treat one peak as a single word to translate. 

So from here on, our focus is on how to represent a *single peak* in a way a model can learn from.

### Transformer

Few papers have been as influential as [Attention Is All You Need](https://arxiv.org/abs/1706.03762). It introduced the **Transformer**, a model architecture built specifically for *translation* tasks, like converting an English sentence into French.

The key insight was **attention**: when predicting the next output word, the model doesn't treat all input words equally. It learns to *attend*, to focus more on the words that are most relevant to the current prediction.

<figure style="width: 50%;">
  <img src="https://figures.academia-assets.com/101672149/figure_012.jpg" alt="Attention visualization">
  <figcaption>In the image above, each word attends to the words before it with different strengths (darker = more attention). So "is" attends to "FBI" because "is" refers to the FBI performing some action.</figcaption>
</figure>


This matters because language isn't a bag of independent words. Words depend on each other, and those dependencies carry meaning. Attention lets the model capture that structure.

The reason this is relevant here is that Casanovo frames de novo sequencing as *translation*. Instead of English -> French, it's spectrum -> peptide. And instead of words attending to words, **peaks attend to other peaks**. A fragment ion at m/z = 600 might carry more meaning in the context of a nearby ion at m/z = 601 (a ~1 Th gap). Attention lets the model discover those relationships automatically from data.


## 3.3 The Representation Problem

Recall that machines don't see mass spectra, they see numbers. In Notebook 2, we turned each spectrum into a list of "scrambled" numbers through binning and hashing. Same idea here, but now we need the model to understand *where* each individual peak lives.

If we want to train a translation model, how does it know if a word is the 1st word or the 600th? From a purely computational view, if we had a bunch of words that we fed (as numbers) into a machine learning model and expected output, the model has no idea which words are where, we need to imbue that knowledge into those numbers somehow.

Now map this to spectra: how does our model know whether a peak is at m/z = 100 or m/z = 600?

"Imbuing" this knowledge is actually not as trivial as it sounds.

Our goal is to do this abstract "translation" from peaks -> peptides right now. In order to do that, we need to represent a) intensity and b) m/z values for our model to read.

For now, we'll set our sights on m/z though.

---

### Attempt 1: Just use the number

M/Z values are already numbers right? Why are we trying to overcomplicate this? Just give a model the peak at $m/z = 600$ for example.

The drawback to this is that we lose the idea that we can "attend" to different peaks. See, just like we measured similarity in the last notebook with cosine similarity, here we use just the dot product. And dotting 2 random peaks at $600 \cdot 100$ for example, won't result in a useful answer. 

*In fact, this is a property of most machine learning models; they can't do subtraction out of the box.* 

They "learn" via matrix multiplication and division. So we'd lose the idea that 600 is ~500 m/z away from 100, because that's simply not a property of our model.

---

### Attempt 2: Binary

Okay, so raw numbers won't work. What about binary? The peak at m/z = 8 would be represented as `1000`:

| m/z | Binary |
|-----|--------|
| 1 | `0001` |
| 2 | `0010` |
| 4 | `0100` |
| 8 | `1000` |

This means that the last bit flips every 1 m/z, the next bit flips every 2 m/z, the next every 4 m/z, and so on. This allows us to try and capture the idea of "close" and "far" peaks. For example, m/z = 6 (`0110`) and m/z = 7 (`0111`) are neighbors.

This is more compact! We only need about 11 bits to represent m/z up to 2000. But binary has its own problem: m/z = 7 (`0111`) and m/z = 8 (`1000`) are *neighbors*, yet their binary representations differ in **every single bit**. The model has no sense of "closeness" because 7 and 8 look as different as 7 and 1000. Again, it cannot add or subtract as we can.

---

### What if we could smooth binary out?

Notice that binary digits flip between 0 and 1 at different rates: the rightmost bit flips every step, the next bit every 2 steps, the next every 4, and so on. What if instead of hard 0/1 flips, we used smooth, continuous oscillations at these different frequencies?

Most machine learning models rely on continuity between positions in order to "train their weights" with calculus. If we can create a representation where nearby m/z values have similar representations, the model can learn to attend to those relationships. 

That's exactly what **sinusoidal positional encoding** does.

---

## 3.4 Sinusoidal Positional Encoding Formula

It doesn't take too much of a leap to think of plain old sine and cosine functions to smooth out this binary representation. In fact, instead of just 11 dimensions (number of bits needed to represent the highest m/z values), we can use as many waves as we want, each with a different frequency. 
- 11 dimensions would mean that we jump a lot across frequencies. That is, if the first bit flips every 1 m/z, the next every 2 m/z, the next every 4 m/z, etc., then the 11th bit would flip every 1024 m/z. That's a huge jump in sensitivity from one bit to the next. What if we want to represent 0.1 m/z differences? Or 2.30 m/z? This is why we use many more dimensions to get a smoother representation across the m/z range.

**The core idea will hold though: we are still encoding m/z values as numbers, some of which flip really fast (sensitive to small m/z changes) and some of which flip really slow (sensitive to large m/z changes).**


 For each m/z value, we'll compute one number per wave and collect them into a vector. That vector *is* the sinusoidal positional encoding of that m/z value.

<details><summary><strong>Sine wave terminology refresher (click to expand)</strong></summary>

$$
y = A\sin\big(B(x - h)\big) + k
$$

$$
\begin{aligned}
\text{Amplitude} &= |A|, \\
\text{Midline} &= k, \\
\text{Period} &= \frac{2\pi}{|B|}, \\
\text{Phase shift} &= 
\begin{cases}
\text{right } h & \text{if } h < 0,\\
\text{left } h & \text{if } h > 0.
\end{cases}
\end{aligned}
$$

The **period** is what we care about most here. It controls how quickly the wave oscillates, just like in binary, and with our "rulers," which we'll get to in a second.

</details>

---

### The idea in one line

For a given m/z value and a wavelength $\lambda_i$, compute:

$$
\sin\!\left(\frac{m/z}{\lambda_i}\right)
$$

That's it. Plug in an m/z, get a number between −1 and 1. Different $\lambda_i$ values give different sensitivities to different m/z ranges. 

If we think of each $\lambda_i$ as a different "ruler" for measuring changes in m/z just like our binary example, then:
- A small $\lambda_i$ makes a fast-oscillating wave (fine ruler), a large $\lambda_i$ makes a slow one (coarse ruler).

Do this for 256 different wavelengths, and you have a 256-dimensional vector encoding one peak.

---

### Choosing the wavelengths $\lambda_i$


The wavelengths $\lambda_i$ are spaced *exponentially* from $\lambda_{\text{min}} = 0.001$ to $\lambda_{\text{max}} = 10{,}000$:

$$
\lambda_i = \frac{\lambda_{\text{min}}}{2\pi} \left( \frac{\lambda_{\text{max}}}{\lambda_{\text{min}}} \right)^{i / (d - 1)}, \quad i = 0, 1, \dots, d-1
$$

Imagine a list of 256 wavelengths increasing exponentially:
$$\lambda = [0.001, \;\dots,\; 10{,}000]$$

Each wavelength is a different "ruler": short wavelengths detect tiny m/z differences (isotopes, noise), long wavelengths distinguish broad spectrum regions. Together they let the model read position at many scales simultaneously.

- $d$ = total encoding dimensions. Casanovo uses $d = 512$.

In [ ]:
def make_wavelengths(n_wavelengths: int, lambda_min: float = 0.001, lambda_max: float = 10000) -> np.ndarray:
    """
    Generates a list of wavelengths following the Casanovo formula:
    ꟛ_i = (ꟛ_min / 2π) * (ꟛ_max / ꟛ_min)^(i / (d_sin - 1))
    """
    wavelengths = np.zeros(n_wavelengths)
    for i in range(n_wavelengths):
        wavelengths[i] = (lambda_min / (2 * np.pi)) * (lambda_max / lambda_min) ** (i / (n_wavelengths - 1))
    return wavelengths

# Let's see what wavelengths look like for a 512-dimensional encoding (256 sin and 256 cosine)
# TODO: Emphasize that the wavelengths are the same, if you have the same number of dims for sin and cosine. 
sin_wavelengths = make_wavelengths(256)
print(f"Number of wavelengths: {len(sin_wavelengths)}")
print(f"Smallest wavelength: {sin_wavelengths[0]:.6f}") # we'll see this later!
print(f"Largest wavelength: {sin_wavelengths[-1]:.6f}") # we'll see this later!

<!-- The reason we're making a list of wavelengths is so that we can do something like 

```python
sin_positional_encoding_600 = sin(600 / wavelengths[0:256])
cos_positional_encoding_600 = cos(600 / wavelengths[256:512])
```

easily. -->

### Full Sinusoidal Positional Encoding Formula

Casanovo uses **both** sine and cosine, which doubles the dimensions to 512. (Why both? We cover the intuition for this later.)

The total dimensions are split into two halves:

$$d_{\text{sin}} = \left\lceil \frac{d}{2} \right\rceil \tag{1}$$

$$d_{\text{cos}} = d - d_{\text{sin}} \tag{2}$$

And the encoded features for some $\text{m/z}$ peak called $m_j$ are:

$$f_i(m/z) = \begin{cases} \sin\left( \frac{m/z}{\lambda_i} \right), & \text{for } i \leq d/2 \\[6pt] \cos\left( \frac{m/z}{\lambda_i} \right), & \text{for } i > d/2 \end{cases} \tag{3}$$

where $\lambda_{\text{max}} = 10{,}000$ and $\lambda_{\text{min}} = 0.001$.


- **For our examples, we'll focus only on the sine half** ($i \leq d/2$, i.e., the first 256 dimensions).

---

<details><summary><strong>Numerical example</strong></summary>

Consider the sine wave at $i = 1$ (second wavelength, out of 6 total for illustration):

$$\sin\!\left(\frac{x}{(0.001/2\pi)(10{,}000/0.001)^{1/2}}\right)$$

Its period:
$$
\text{Period} = \frac{2\pi}{B}, \quad B = \frac{1}{\left(\frac{0.001}{2\pi}\right)\left(\frac{10000}{0.001}\right)^{1/2}} = 2\pi \cdot \frac{0.001}{2\pi} \cdot 3162.277 \approx 3.162
$$

So this wave completes one full cycle every **~3.16 m/z units**.

![A single period plotted for this sine wave](Images/desmos_1.png)

</details>

In [ ]:
# Define positional encoding function used by Casanovo
def positional_encoding(m_z, d_model=512):
    """
    Encode a single m/z value into a d_model-dimensional vector.

    If d_model is even, then the wavelengths are shared between sine and cosine. 
    If d_model is odd, then we will have one more sine wavelength than cosine wavelength.
    
    d_sin: The first ⌈d_model / 2⌉ dimensions represent the sine encoding
    d_cos: The last d_model - d_sin dimensions represent the cosine encoding
    """
    encoding = np.zeros(d_model)
    d_sin = math.ceil(d_model / 2)
    d_cos = d_model - d_sin
    sin_wavelengths = make_wavelengths(d_sin)
    cos_wavelengths = sin_wavelengths if d_cos == d_sin else make_wavelengths(d_cos)
    for d in range(d_sin):
        wavelength = sin_wavelengths[d]
        encoding[d] = np.sin(m_z / wavelength)           # First half: sine
    for d in range(d_sin, d_model):
        wavelength = cos_wavelengths[d - d_sin]
        encoding[d] = np.cos(m_z / wavelength)           # Second half: cosine
    return encoding



And we're done! We can just plug in `positional_encoding(600)` and get a 512-dimensional vector representing the peak at m/z = 600.

## 3.5 Low-Dimensional Examples and Visualizations:

### Example: encoding $m/z=600$

To build intuition, we'll use $d = 6$ dimensions (instead of Casanovo's 256 for sine) and trace exactly what the formula produces.

Recall the wavelength formula from [3.4](#34-sinusoidal-positional-encoding-formula):

$$\lambda_i = \frac{\lambda_{\min}}{2\pi} \left(\frac{\lambda_{\max}}{\lambda_{\min}}\right)^{i/(d-1)}, \quad i = 0, 1, \dots, 5$$

with $\lambda_{\min} = 0.001$, $\lambda_{\max} = 10{,}000$. Each $\sin(m/z \,/\, \lambda_i)$ has **period** $2\pi \times \lambda_i$. This implies the $2\pi$ in the wavelength formula isn't magic, it's just pre-compensating for this factor so that the periods land on the clean exponential grid from $0.001$ to $10{,}000$ rather than on $0.001/2\pi$ to $10{,}000/2\pi$.


<details><summary><strong>How do we compute the period of this wave?</strong></summary>

Recall from above that for a general sine wave:

$$\sin(Bx) \quad \text{has period} \quad \frac{2\pi}{B}$$

In our case the argument is $\dfrac{m/z}{\lambda_i}$, so $B = \dfrac{1}{\lambda_i}$, giving:

$$\text{period} = \frac{2\pi}{1/\lambda_i} = 2\pi \times \lambda_i$$


---
</details>

First, we'll use our function from earlier. If you'll recall, we designed it so that $\lambda_{\min}$, $\lambda_{\max}$, and *d* can be passed in as arguments.

In [ ]:
# Calculating wavelengths for a 6-dimensional, purely sinusoidal encoding (no cosine)

six_wavelengths = make_wavelengths(n_wavelengths=6, lambda_min=0.001, lambda_max=10_000) # Told you we'd see this again!

print(f"Number of wavelengths: {len(six_wavelengths)}")
print(f"λ₀: {six_wavelengths[0]:.5f}") 
print(f"λ₁: {six_wavelengths[1]:.5f}") 
print(f"λ₂: {six_wavelengths[2]:.5f}") 
print(f"λ₃: {six_wavelengths[3]:.5f}") 
print(f"λ₄: {six_wavelengths[4]:.5f}")
print(f"λ₅: {six_wavelengths[5]:.5f}") 


### Computing the six wavelengths:

$$
\begin{aligned}
\lambda_0 &= \frac{0.001}{2\pi} \times (\frac{0.001}{10,000})^{0/5} = \frac{0.001}{2\pi} \times 1 &&= 0.00016 \\
\lambda_1 &= \frac{0.001}{2\pi} \times (\frac{0.001}{10,000})^{1/5} = \frac{0.001}{2\pi} \times 25.11886 &&= 0.00400 \\
\lambda_2 &= \frac{0.001}{2\pi} \times (\frac{0.001}{10,000})^{2/5} = \frac{0.001}{2\pi} \times 630.95734 &&= 0.10042 \\
\lambda_3 &= \frac{0.001}{2\pi} \times (\frac{0.001}{10,000})^{3/5} = \frac{0.001}{2\pi} \times 15848.93192 &&= 2.52244 \\
\lambda_4 &= \frac{0.001}{2\pi} \times (\frac{0.001}{10,000})^{4/5} = \frac{0.001}{2\pi} \times 398107.17055 &&= 63.36072 \\
\lambda_5 &= \frac{0.001}{2\pi} \times (\frac{0.001}{10,000})^{5/5} = \frac{0.001}{2\pi} \times 10000000 &&= 1591.54943
\end{aligned}
$$

Now we see that our function from earlier agrees with our calculation

**The resulting periods** $2\pi \times \lambda_i$ land exactly on the exponential grid too ($\times \sim 25$ each time):

| $i$ | $\lambda_i$ | Period $= 2\pi \times \lambda_i$ | Sensitivity |
|-----|-------------|----------------------------------|-------------|
| 0 | 0.00016 | 0.001 m/z | Below the resolving power for most instruments. No real signal at this resolution |
| 1 | 0.00400 | 0.025 m/z | Isotopic spacing for highly charged ions (e.g. z=40 → 1/40 Da apart) |
| 2 | 0.10042 | 0.631 m/z | Isotopic spacing for singly/doubly charged ions (~0.5–1 Da) |
| 3 | 2.52244 | 15.849 m/z | Mass difference between different amino acids  (amino acid masses range ~57–186 Da) |
| 4 | 63.36072 | 398.107 m/z | The mass of a small, doubly charged peptide ( m = 800, z = 2 ) |
| 5 | 1591.54943 | 10,000 m/z | Above the m/z range for most instruments (distinguishes between extremely broad spectral regions) |


Now, let's see what a peak at m/z = 600 would look like in six dimensions!


In [ ]:
# Encoding one m/z value in six dimensions, using the six wavelengths we just calculated.
m_z = 600.0

# The Casanovo encoding function we wrote earlier automatically calculates both sin and cos components, which is why we're using this loop instead
print("Positional encoding for m/z=600.0 with 6 dimensions:")
print("[" + ", ".join(f"{x:.5f}" for x in [math.sin(m_z / wavelength ) for wavelength in six_wavelengths]) + "]") 


**Sinusoidal Positional Encoding for a peak at $m/z = 600$:**

$$\mathbf{f}(600) = \left[\sin\!\left(\frac{600}{\lambda_i}\right)\right]_{i=0}^{5} = [\approx 0,\; 0.42445,\; -0.39186,\; -0.78066,\; -0.04480,\; 0.36812]$$

> $i=0$ cycles so rapidly at this scale that its value is effectively arbitrary noise, as you'll see in the video and in the table below




### Visualization: encoding $m/z=600$

Note that sinusoidal positional encoding isn't just a math formula you plug numbers into or memorize. It has deep geometric intuition which we've somewhat introduced, and the best way to understand it is to see it in action whilst encoding a peak at $m/z = 600$:

In [ ]:
from IPython.display import Video
Video('Videos/SinusoidalPE.mp4', embed=True, width=800)

**Let's jump back into our high-dimensional space with 256 sine, 256 cosine dimensions, and see what the encoding for $m/z = 600$ looks like:**

In [ ]:
encode_600 = positional_encoding(600)

In [ ]:
# Visualize the encodings
fig, ax = plt.subplots(1, 1, figsize=(10, 4))

ax.scatter(range(len(encode_600)), encode_600, s=5, alpha=0.7)
ax.axvline(x=256, color='red', linestyle='--', alpha=0.5, label='sine | cosine boundary')
ax.set_title("Positional Encoding — m/z = 600")
ax.set_xlabel("Dimension (i)")
ax.set_ylabel("Encoded Value")
ax.legend()

plt.tight_layout()
plt.show()

**Some Corollaries:**
- The left side of each plot (fine wavelengths) looks like noise. This is exactly what we were seeing in our low dimensional example, and in the video. These waves are oscillating so fast that small m/z changes cause large value changes. 
- The right side (coarse wavelengths) shows smooth patterns - these change only for large m/z differences


### Example: encoding $m/z=600 \to m/z=601$


$$\mathbf{f}(601) = [\approx 0,\; 0.99838,\; -0.13037,\; -0.47880,\; -0.06056,\; 0.36871]$$


| $i$ | Period | Cycles in 1 m/z | $\sin$ at 600 | $\sin$ at 601 | $\Delta$ |
|-----|--------|-----------------|---------------|---------------|----------|
| 0 | 0.001 | 1000.00 | ≈ 0 | ≈ 0 | 0: Completes 1000 full cycles: value is arbitrary, not meaningful |
| 1 | 0.025 | 39.81 | 0.42445 | 0.99838 | **+0.574**: wraps ~40×: better but not uniquely locating |
| 2 | 0.631 | 1.58 | −0.39186 | −0.13037 | **+0.261**: just over one full cycle, still changes a lot |
| 3 | 15.849 | 0.063 | −0.78066 | −0.47880 | **+0.302:** 6% of a cycle; still meaningfully shifts the vector |
| 4 | 398.107 | 0.003 | −0.04480 | −0.06056 | **−0.016:** barely 0.3% of a cycle; nearly identical |
| 5 | 10,000 | 0.0001 | 0.36812 | 0.36871 | **+0.001**: 600 and 601 are indistinguishable here, by design |

</br>

**Recall that the model has no explicit subtraction operator to compare two peaks**. 

The model compares peaks via dot products between their encoding vectors, not by explicitly computing differences. Recall from above that it cannot do subtraction out of the box. This implies that it can't compute $\mathbf{f}(601) - \mathbf{f}(600)$ and read off the difference. Rather, the coarse waves ensure that nearby peaks have high dot product similarity; the fine waves are what break ties between peaks that are close but not identical.

### Summary

The broad idea is that waves ($i=4,5$) are what **locate** a peak in the spectrum. They change slowly enough to be monotonically informative across large m/z ranges. Fine waves ($i=1,2$) change too rapidly to locate, but are sensitive enough to distinguish nearby peaks once the coarse waves have narrowed the region. 


So when we relate this back to the question of "*how does a model know if a peak is at m/z = 100 or 600?*", the answer is that the coarse waves will be very different between those two m/z values, giving the model a strong signal about their relative positions. The fine waves will then provide additional detail about the local structure of the spectrum around those positions.


> **Desmos link for all graphs:** https://www.desmos.com/calculator/he8fjmng6v 
 - (note that we only made the coarser waves more visible by default, but you can toggle the fine ones on/off)

### Visualization: encoding $m/z = 600 \to m/z = 601$

Here, I hope to show you the following:

| Dimension Range | Wavelength | Information Captured |
|-----------------|------------|---------------------|
| 0–50 (sine), 256–306 (cosine) | Short | High-resolution, changes rapidly with small m/z shifts |
| 200–255 (sine), 456–511 (cosine) | Long | Low-resolution, stable across nearby m/z values |

This should align with the intuition we've been building up - the lower dimensions are fine, the higher dimensions are coarse. This just puts it into 256 dimensions instead of 6.


In [ ]:
encode_601 = positional_encoding(601)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))


axes[0].scatter(range(0, 57), encode_600[0:57], label='m/z = 600', alpha=0.7)
axes[0].scatter(range(0, 57), encode_601[0:57], label='m/z = 601', alpha=0.7)
axes[0].set_title("Fine Wavelengths (dims 0-56)\nCorrectly distinguishes SMALL m/z differences")
axes[0].set_xlabel("Dimension (i)")
axes[0].set_ylabel("Value")
axes[0].legend(loc='upper right')

axes[1].scatter(range(200, 256), encode_600[200:256], label='m/z = 600', alpha=0.7)
axes[1].scatter(range(200, 256), encode_601[200:256], label='m/z = 601', alpha=0.7)
axes[1].set_title("COARSE Wavelengths (dims 200-255)\nNearly identical for nearby m/z")
axes[1].set_xlabel("Dimension (i)")
axes[1].set_ylabel("Value")
axes[1].legend(loc='upper right')

plt.suptitle("Zooming in on the fine vs coarse wavelengths\n(m/z = 600 vs m/z = 601)", fontsize=16)
plt.tight_layout()
plt.show()


## 3.6 Why Both Sine AND Cosine?

Throughout this explanation, you may have seen various references to "we'll just focus on the sine part of the equation for now" (see full positional encoding equation image for reference). 

The reason we use both sine and cosine is elegant: it gives the model unique vectors for all positions, and it allows the model to jump from 1 m/z value to another m/z value and compute the difference (recall, model's can't subtract!) 

<details><summary>This part is the most math-heavy we'll get, so it's optional</summary>

### "How does a model know if a peak is at m/z = 100 or 600?"

Consider what happens in the imaginary case wherein we get the exact same encoding for both $m/z = 100$ and $m/z = 600$ for the sine waves.

That is, $\sin(100/\lambda) = \sin(600/\lambda)$. This is not only possible, but it will happen. The idea is that sine waves are not one-to-one functions; they repeat in every single period. So it's inevitable that some pairs of m/z values will produce identical sine encodings, making them indistinguishable if we only had sine.

For a trivial example:

$$\sin(0) = \sin(2\pi) = \sin(4\pi) = 0$$

There are two complementary fixes to this problem.

1. **Within a single period**, adding cosine resolves all ambiguity: $(\sin\theta, \cos\theta)$ uniquely identifies any angle in $[0, 2\pi)$. I encourage you to try to find two different angles that have the same sine and cosine values. You won't be able to, *because the pair together is a unique coordinate on the unit circle*.

2. **Across periods**, cosine alone is not enough. Any two m/z values differing by exactly one full period will still collide:
$$\mathbf{f}(0) = [\sin(0),\ \cos(0)] = [0,\ 1]$$
$$\mathbf{f}(2\pi) = [\sin(2\pi),\ \cos(2\pi)] = [0,\ 1]$$

This is where many wavelengths come in. 256 of each sine and cosine wave, to be exact. Two m/z values that collide at one wavelength would need to collide at every wavelength simultaneously to be truly indistinguishable. Since the wavelengths are spaced exponentially and are irrational multiples of each other, this never happens over any realistic m/z range.

### Relative Positions

Imagine this sine and cosine embedding as a set of dials. 

Moving forward by a specific distance (like a $\Delta$ of 1.0033 Da between isotopes) is equivalent to rotating a dial by a specific number of clicks. It doesn't matter if you are at $m/z$ 100 or 800; rotating that dial by the same amount always moves you the exact same distance. Rotate it twice, and you move twice as far. This allows the model to natively 'feel' the relative spacing between peaks, *without* memorizing absolute numbers

**Full proof:** [Linear Relationships in Positional Encoding](https://blog.timodenk.com/linear-relationships-in-the-transformers-positional-encoding/)
</details>


In [ ]:
encode_100 = positional_encoding(100)

In [ ]:
# Compare LOW-resolution dimensions (coarse wavelengths: 200-255)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(range(200, 256), encode_600[200:256], label='m/z = 600', alpha=0.7)
axes[0].scatter(range(200, 256), encode_601[200:256], label='m/z = 601', alpha=0.7)
axes[0].set_title("COARSE Wavelengths (dims 200-255)\nShould be similar for nearby m/z")
axes[0].set_xlabel("Dimension (i)")
axes[0].set_ylabel("Value")
axes[0].legend()

axes[1].scatter(range(200, 256), encode_600[200:256], label='m/z = 600', alpha=0.7)
axes[1].scatter(range(200, 256), encode_100[200:256], label='m/z = 100', alpha=0.7, color='red')
axes[1].set_title("COARSE Wavelengths (dims 200-255)\nShould be different for distant m/z")
axes[1].set_xlabel("Dimension (i)")
axes[1].set_ylabel("Value")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Compare HIGH-resolution dimensions (fine wavelengths: 0-56)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(range(0, 57), encode_600[0:57], label='m/z = 600', alpha=0.7)
axes[0].scatter(range(0, 57), encode_601[0:57], label='m/z = 601', alpha=0.7)
axes[0].set_title("Fine Wavelengths (dims 0-56)\nCan distinguish SMALL m/z differences")
axes[0].set_xlabel("Dimension (i)")
axes[0].set_ylabel("Value")
axes[0].legend()

axes[1].scatter(range(0, 57), encode_600[0:57], label='m/z = 600', alpha=0.7)
axes[1].scatter(range(0, 57), encode_100[0:57], label='m/z = 100', alpha=0.7, color='red')
axes[1].set_title("Fine Wavelengths (dims 0-56)\nHighly variable for all m/z")
axes[1].set_xlabel("Dimension (i)")
axes[1].set_ylabel("Value")
axes[1].legend()

plt.tight_layout()
plt.show()

### Visualzing the Encodings
Here we use scan 9970 from the same mzML file as earlier. We plot the encoded m/z (512 dimensions) and intensity (512 dimensions) on the y-axis for each of the top 200 most intense peaks in the spectrum. The color represents the encoded value (between -1 and 1). The smooth, continuous bands across peaks show that Casanovo encoding preserves similarity within the spectrum: peaks that are close in m/z remain close in the encoded space across many dimensions, especially at the min and max m/z and intensity wavelengths.

In [ ]:
ms2_scan = get_MS2_object('Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML', 5423)
print(f"Let's see what scan 5423 looks like natively:")
plot_MS2(ms2_scan).show()
print(f"It has {len(ms2_scan.mz)} peaks, all of which we'll encode below")


import matplotlib.pyplot as plt
import numpy as np

# 1. Inject a blank gap into the data matrix
gap_size = 12
gap = np.full((gap_size, n_peaks), np.nan)
# Stack Sine (0-255), the empty gap, and Cosine (256-511)
E_split = np.vstack([E[:256, :], gap, E[256:, :]])

# ------------------------------------------------------------------
# Clean Styling Configuration
# ------------------------------------------------------------------
fine_color = '#2563eb'
coarse_color = '#2e7d32'
bg = '#f8fafc'

fig, ax = plt.subplots(figsize=(18, 9), facecolor=bg)
# Increased left margin to 0.16 to give the brackets and numbers plenty of room
plt.subplots_adjust(left=0.16, right=0.88, top=0.90, bottom=0.15)

# Tell the colormap to render our NaN gap as the background color
cmap = plt.get_cmap('magma').copy()
cmap.set_bad(color=bg)

im = ax.imshow(
    E_split,
    aspect='auto',
    vmin=-1,
    vmax=1,
    interpolation='nearest',
    cmap=cmap,
)

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.08)
cbar.ax.set_ylabel('Encoded value (-1 to 1)', fontsize=16, fontweight='bold', rotation=270, labelpad=30, va='center')
cbar.ax.tick_params(labelsize=10)

# Add Sine/Cosine labels cleanly to the outside right
ax.text(1.02, 128, 'SINE ENCODING', transform=ax.get_yaxis_transform(),
        rotation=-90, va='center', ha='left', fontsize=12, fontweight='bold', color='#444444')
ax.text(1.02, 384 + gap_size, 'COSINE ENCODING', transform=ax.get_yaxis_transform(),
        rotation=-90, va='center', ha='left', fontsize=12, fontweight='bold', color='#444444')

# 2. Draw clean brackets on the outside left (pointing inward, clearing the numbers)
def draw_left_bracket(ax, y0, y1, text, color):
    trans = ax.get_yaxis_transform()
    
    # Coordinates for an inward-facing bracket [
    x_tips = -0.045     # Tips sit just outside the dimension numbers
    x_bar = -0.060      # Spine is pushed further left
    x_mid_tick = -0.070 # Center tick pointing outward towards the text
    text_x = -0.085     # Text is safely in the margin
    
    y_start = y0 - 0.5
    y_end = y1 + 0.5
    y_mid = (y_start + y_end) / 2
    
    # Draw the main spine
    ax.plot([x_bar, x_bar], [y_start, y_end], color=color, lw=2.5, clip_on=False, transform=trans)
    # Draw the top and bottom tips pointing INWARD
    ax.plot([x_bar, x_tips], [y_start, y_start], color=color, lw=2.5, clip_on=False, transform=trans)
    ax.plot([x_bar, x_tips], [y_end, y_end], color=color, lw=2.5, clip_on=False, transform=trans)
    # Draw a center tick pointing OUTWARD to the label
    ax.plot([x_bar, x_mid_tick], [y_mid, y_mid], color=color, lw=2.5, clip_on=False, transform=trans)
    
    # Label with a soft background
    bbox_props = dict(boxstyle="round,pad=0.3", fc=color, ec='none', alpha=0.15)
    ax.text(text_x, y_mid, text, transform=trans,
            rotation=90, ha='center', va='center', color=color, 
            fontweight='900', fontsize=13, bbox=bbox_props)
    
# Draw the brackets
draw_left_bracket(ax, 0, 56, 'Fine', fine_color)
draw_left_bracket(ax, 200, 255, 'Coarse', coarse_color)
draw_left_bracket(ax, 256 + gap_size, 312 + gap_size, 'Fine', fine_color)
draw_left_bracket(ax, 456 + gap_size, 511 + gap_size, 'Coarse', coarse_color)

# 3. Axes Labels and Formatting
ax.set_xlabel('Peaks (sorted by m/z)', fontsize=16, fontweight='bold', labelpad=10)
# Increased labelpad drastically to jump over the numbers AND the brackets
ax.set_ylabel('Encoding dimension (0-511)', fontsize=16, fontweight='bold', labelpad=110)

ax.set_title(
    'Scan 5423 (m/z only) positional encoding',
    fontsize=16,
    fontweight='bold',
    pad=15
)

# X-ticks
xticks = np.linspace(0, n_peaks - 1, 12, dtype=int)
ax.set_xticks(xticks)
ax.set_xticklabels([f'{mz_sorted[i]:.1f}' for i in xticks], rotation=30, ha='right', fontsize=10)

# Y-ticks (Numbers stay put, brackets now encircle them)
y_tick_locations = [0, 56, 128, 200, 256 + gap_size, 312 + gap_size, 384 + gap_size, 456 + gap_size, 511 + gap_size]
y_tick_labels =    ['0', '56', '128', '200', '256',          '312',          '384',          '456',          '511']

ax.set_yticks(y_tick_locations)
ax.set_yticklabels(y_tick_labels, fontsize=10)

# Extend the y-limit slightly to account for the new rows
ax.set_ylim(511.5 + gap_size, -0.5)

# Clean up spines
for spine in ['top', 'right', 'bottom', 'left']:
    ax.spines[spine].set_linewidth(1.0)
    ax.spines[spine].set_color('#333333')

plt.show()

Notice the stark difference in frequency across the encoding dimensions. In the 'Fine' ranges, the values (colors) oscillate rapidly to capture local peak-to-peak differences. 

Meanwhile, the 'Coarse' dimensions act as long waves, maintaining their encoded values over spans of hundreds of m/z to anchor the absolute position.

## 3.7 Intensity Encoding

Notice that we've covered positional encoding with m/z very clearly, but we haven't covered the other input to a spectrum: intensity.

The Casanovo authors mention that we can either positionally encode intensity in the same way (since the only input is raw values like 1.319) as m/z, or we can throw them into the model through a *learnable linear layer* (imagine intensity encodings that the model itself updates) and encode them that way.